## Resultados, documentação, governança e validação da Gold

In [0]:
%sql
SELECT
  'VoeBem Analytics' AS projeto,
  'Resultados da camada Gold' AS objetivo;

### 1. Tabelas Gold

In [0]:
%sql
SELECT
  table_name,
  table_type,
  comment
FROM voebem.information_schema.tables
WHERE table_schema = 'gold'
  AND table_name IN (
    'dim_aeroporto',
    'fato_voos',
    'obt_voos'
  )
ORDER BY table_name;

### 2. Quantidade de registros por tabela


In [0]:
%sql
SELECT
  'dim_aeroporto' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.dim_aeroporto

UNION ALL

SELECT
  'fato_voos' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'obt_voos' AS tabela,
  COUNT(*) AS registros
FROM voebem.gold.obt_voos

ORDER BY registros DESC;

### 3. Evolução da quantidade de registros


In [0]:
%sql
SELECT
  'Bronze VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos' AS camada,
  COUNT(*) AS registros
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos' AS camada,
  COUNT(*) AS registros
FROM voebem.gold.obt_voos

ORDER BY
  CASE camada
    WHEN 'Bronze VRA' THEN 1
    WHEN 'Silver VRA' THEN 2
    WHEN 'Gold Fato Voos' THEN 3
    WHEN 'Gold OBT Voos' THEN 4
  END;

Databricks visualization. Run in Databricks to view.

### 4. Documentação das colunas Gold


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS total_colunas,
  COUNT(comment) AS colunas_comentadas,
  COUNT(*) - COUNT(comment) AS sem_comentario,
  ROUND(
    100.0 * COUNT(comment) / COUNT(*),
    2
  ) AS percentual_documentado
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

### 5. Quantidade de colunas por tabela


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS quantidade_colunas
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY quantidade_colunas DESC;

### 6. Estrutura da OBT


In [0]:
%sql
SELECT
  ordinal_position,
  column_name,
  data_type,
  comment
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name = 'obt_voos'
ORDER BY ordinal_position;

### 7. Tags das tabelas Gold


In [0]:
%sql
SELECT
  table_name AS tabela,
  tag_name,
  tag_value
FROM voebem.information_schema.table_tags
WHERE schema_name = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
ORDER BY
  table_name,
  tag_name;

### 8. Quantidade de tags por tabela


In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS quantidade_tags
FROM voebem.information_schema.table_tags
WHERE schema_name = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

### 9. Minutos recuperados x atraso na chegada


In [0]:
%sql
SELECT
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END AS categoria,

  COUNT(*) AS quantidade_voos

FROM voebem.gold.obt_voos

GROUP BY
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END

ORDER BY quantidade_voos DESC;

### 10. Validação específica da descrição da métrica


In [0]:
%sql
SELECT
  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
  ) AS recuperaram_tempo,

  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
      AND atraso_chegada_min > 15
  ) AS recuperaram_mas_chegaram_atrasados,

  COUNT(*) FILTER (
    WHERE minutos_recuperados > 0
      AND atraso_chegada_min <= 15
  ) AS recuperaram_e_chegaram_com_ate_15_min,

  ROUND(
    100.0 *
    COUNT(*) FILTER (
      WHERE minutos_recuperados > 0
        AND atraso_chegada_min > 15
    )
    /
    NULLIF(
      COUNT(*) FILTER (
        WHERE minutos_recuperados > 0
      ),
      0
    ),
    2
  ) AS percentual_recuperaram_mas_atrasaram

FROM voebem.gold.obt_voos;

### 11. Validação de partida_pontual para voos cancelados


In [0]:
%sql
SELECT
  situacao_voo,
  partida_pontual,
  COUNT(*) AS quantidade_voos
FROM voebem.gold.obt_voos
WHERE situacao_voo IN ('CANCELADO', 'REALIZADO')
GROUP BY
  situacao_voo,
  partida_pontual
ORDER BY
  situacao_voo,
  partida_pontual;

Databricks visualization. Run in Databricks to view.

### 12. Validação de mes_referencia


In [0]:
%sql
SELECT
  CASE
    WHEN mes_referencia IS NULL THEN 'NULL'
    ELSE 'Preenchido'
  END AS status_mes_referencia,
  COUNT(*) AS quantidade_voos
FROM voebem.gold.obt_voos
GROUP BY
  CASE
    WHEN mes_referencia IS NULL THEN 'NULL'
    ELSE 'Preenchido'
  END
ORDER BY status_mes_referencia;

### 13. Atraso médio por companhia


In [0]:
%sql
SELECT
  icao_empresa,
  COUNT(*) AS quantidade_voos,
  ROUND(AVG(atraso_partida_min), 2) AS atraso_medio_partida,
  ROUND(AVG(atraso_chegada_min), 2) AS atraso_medio_chegada
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
GROUP BY icao_empresa
HAVING COUNT(*) >= 100
ORDER BY atraso_medio_chegada DESC;

### 14. Aeroportos com maior quantidade de atrasos


In [0]:
%sql
SELECT
  icao_destino,
  COUNT(*) AS voos_atrasados
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
  AND atraso_chegada_min > 15
GROUP BY icao_destino
ORDER BY voos_atrasados DESC
LIMIT 15;

Databricks visualization. Run in Databricks to view.

### 15. Rotas com maior quantidade de atrasos


In [0]:
%sql
SELECT
  CONCAT(icao_origem, ' → ', icao_destino) AS rota,
  COUNT(*) AS voos_atrasados
FROM voebem.gold.obt_voos
WHERE situacao_voo = 'REALIZADO'
  AND atraso_chegada_min > 15
GROUP BY
  icao_origem,
  icao_destino
ORDER BY voos_atrasados DESC
LIMIT 15;

### 16. Lineage relacionado à OBT


In [0]:
%sql
SELECT
  event_time,
  source_table_full_name,
  target_table_full_name,
  entity_type
FROM system.access.table_lineage
WHERE
  target_table_full_name = 'voebem.gold.obt_voos'
  OR source_table_full_name = 'voebem.gold.obt_voos'
ORDER BY event_time DESC;

### 17. Resumo final do projeto


In [0]:
%sql
SELECT
  'Bronze VRA' AS etapa,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA',
  COUNT(*)
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos',
  COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos',
  COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 1 — Registros por tabela

In [0]:
%sql
SELECT 'dim_aeroporto' AS tabela, COUNT(*) AS registros
FROM voebem.gold.dim_aeroporto

UNION ALL

SELECT 'fato_voos', COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT 'obt_voos', COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 2 — Bronze → Silver → Gold

In [0]:
%sql
SELECT
  'Bronze VRA' AS camada,
  COUNT(*) AS registros
FROM voebem.bronze.vra

UNION ALL

SELECT
  'Silver VRA',
  COUNT(*)
FROM voebem.silver.vra

UNION ALL

SELECT
  'Gold Fato Voos',
  COUNT(*)
FROM voebem.gold.fato_voos

UNION ALL

SELECT
  'Gold OBT Voos',
  COUNT(*)
FROM voebem.gold.obt_voos;

Databricks visualization. Run in Databricks to view.

### Gráfico 3 — Documentação

In [0]:
%sql
SELECT
  table_name AS tabela,
  COUNT(*) AS total_colunas,
  COUNT(comment) AS colunas_comentadas,
  ROUND(100.0 * COUNT(comment) / COUNT(*), 2) AS percentual_documentado
FROM voebem.information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN (
    'obt_voos',
    'fato_voos',
    'dim_aeroporto'
  )
GROUP BY table_name
ORDER BY tabela;

Databricks visualization. Run in Databricks to view.

### Gráfico 4 — minutos_recuperados

## Análisis técnico de la validación Gold

### 1. Arquitectura de la capa Gold: tres tablas, tres propósitos

La Gold no es un espejo — es donde nacen las reglas de negocio. Tres tablas con
roles distintos:

| Tabla | Filas | Columnas | Propósito |
|---|---|---|---|
| `dim_aeroporto` | 396 | 7 | Dimensión de aeropuerto, sirve origen Y destino |
| `fato_voos` | 1.014.664 | 31 | Facto con reglas de negocio, sin nombres resueltos |
| `obt_voos` | 1.014.664 | 39 | One Big Table desnormalizada, lista para consumo por IA |

El flujo: `silver.vra` → `fato_voos` (reglas de negocio + dedup) → `obt_voos`
(join con `dim_aeroporto` para resolver nombres).

### 2. Reglas de validación aplicadas en la Gold

#### 2.1 Deduplicación exacta (fato_voos, única exclusión de filas)

`ROW_NUMBER() OVER (PARTITION BY icao_empresa, numero_voo, codigo_di,
codigo_tipo_linha, icao_origem, icao_destino, partida_prevista, partida_real,
chegada_prevista, chegada_real, situacao_voo ORDER BY _ingerido_em)`

Elimina **41 filas byte-idénticas** — la misma etapa publicada dos veces por la
ANAC. Es la **única exclusión de filas en todo el pipeline Bronze → Gold**.
Grano esperado: 1.014.705 - 41 = 1.014.664. Resultado: 1.014.664.

Impacto: sin esta deduplicación, 41 etapas contarían doble, inflando vuelos,
cancelamientos y atrasos simultaneamente.

#### 2.2 Decisión de cuarentena: 5 categorías, 4 mantenidas, 1 eliminada

El marco-06 diagnosticó 213.543 registros problemáticos. Las decisiones están
documentadas en el SQL del fato_voos y son el núcleo de la validación Gold:

| Categoría | Registros | Decisión | Justificación |
|---|---|---|---|
| Aeropuerto fuera del cadastro ANAC | 105.932 | Mantenido | No es inválido — es aeropuerto extranjero. `dim_aeroporto` los cubre con fallback textual |
| Voo sin horario previsto | 30.800 | Mantenido | El voo aconteceu; la métrica de atraso queda NULL (se autoexcluye de promedios) |
| Atraso fuera de faixa plausible | 778+823 | Línea mantenida, métrica anulada | Atrasos de 44.855 min (31 días) son error de fecha, no operación. `atraso_fora_de_faixa` registra el motivo |
| Empresa sin cadastro | 69 | Mantenido con nombre de fallback | `COMPANHIA NAO CADASTRADA (ICAO)` — no vacío, no NULL |
| Duplicata exacta | 41 | **Removida** | Misma etapa publicada dos veces; contarla doble infla todo |

El principio: **la línea cuenta como voo aunque la métrica no se pueda calcular**.
Null es ausencia de información, no cero. Zerar sería mentir.

#### 2.3 Validación de rango plausible en atrasos

`(v.atraso_partida_min < -120 OR v.atraso_partida_min > 1440)` — menos de 2 horas
de antelación o más de 24 horas de atraso se considera error de fecha en la fuente.
La métrica se anula (NULL), la fila se mantiene y `atraso_fora_de_faixa = true`
registrando el motivo. Esto evita que valores absurdos (31 días de "atraso")
distorsionen promedios.

#### 2.4 Revisión de descripciones generadas por IA (gobernanza)

La validación más sofisticada de la Gold no es sobre los datos — es sobre los
**metadatos**. El notebook `09_governanca_gold` documenta un caso crítico:

Un rascunho generado por IA para `minutos_recuperados` decía:
> *"Valor positivo indica que el voo llegó adelantado."*

La validación contra el dato reveló: **164.895 voos recuperaron tiempo y aún así
llegaron atrasados** (75.082 con más de 15 minutos). La descripción correcta:
> *"Positivo significa que llegó menos atrasada de lo que salió, y NO que llegó
> en horario."*

La diferencia entre "recuperó" y "resolvió" es la diferencia entre una decisión
correcta y incorrecta de negocio. El método de validación: **preguntar al dato**,
no releer la frase.

También se validaron:
- `partida_pontual`: 29.140 voos cancelados tienen NULL (nunca se cuentan como atraso)
- `partida_pontual`: 31.679 voos realizados sin horario previsto también NULL
- `mes_referencia`: 30.798 NULL (voos sin horario previsto)

#### 2.5 Auditoría de gobernanza

- **Cobertura de comentarios**: 100% en las 3 tablas (7 + 31 + 39 = 77 columnas)
- **Tags**: 5 por tabla (`camada`, `dominio`, `grao`, `padrao`, `consumo`) —
  `consumo` distingue `bi` (fato/dim) de `genie` (obt), señalando el consumidor
- **Lineage**: verificado vía `system.access.table_lineage` — sin código de
  trazabilidad, el Unity Catalog registra solo la cadena
  `archivo CSV → bronze → silver → gold`

#### 2.6 Evolución de contagem Bronze → Gold

| Capa | Registros | Diferencia |
|---|---|---|
| Bronze VRA | 1.014.705 | — |
| Silver VRA | 1.014.705 | 0 (espejo) |
| Gold Fato Voos | 1.014.664 | -41 (dedup) |
| Gold OBT Voos | 1.014.664 | 0 (sin filtro) |

La única pérdida de filas en todo el pipeline es la deduplicación de 41
registros, documentada y esperada.

### 3. Decisiones técnicas relevantes

- **Dimensión nace del fato, no del cadastro**: `dim_aeroporto` se construye con
  `DISTINCT icao` de `silver.vra` (origen + destino), no copiando
  `silver.aerodromos`. Copiar el cadastro daría 496 líneas y dejaría 218
  aeropuertos del fato huérfanos (los extranjeros). La dimensión existe para
  servir al fato.

- **Defensa contra duplicación de ICAO**: `ROW_NUMBER() OVER (PARTITION BY icao
  ORDER BY nome)` en el cadastro. Si la ANAC republica con ICAO repetido, el join
  no multiplica líneas del fato. Hoy son 496/496.

- **Dimensión degenerada para IA**: compañía, código DI y tipo de línea se
  guardan como código + descripción en el propio fato. Para un consumidor LLM,
  cada join a menos es un error a menos. El hecho no está en 3NF — está
  optimizado para lectura directa.

- **Fallback textual obligatorio**: ninguna columna que la IA lea puede ser NULL.
  `COALESCE(c.nome, 'AEROPORTO FORA DO CADASTRO ANAC (' || a.icao || ')')` — el
  aeropuerto extranjero aparece con su código, no como vacío.

- **Regla de oro de la OBT**: ninguna columna de código sin su descripción al
  lado. `icao_empresa` → `nome_companhia`, `codigo_di` → `descricao_di`,
  `icao_origem` → `nome_aeroporto_origem`. El LLM lee nombres, no códigos ICAO.

- **Deduplicación de empresa con preferencia**: `ROW_NUMBER() OVER (PARTITION BY
  icao ORDER BY CASE WHEN situacao='ATIVA' THEN 0 ELSE 1 END, razao_social)` —
  si la misma empresa aparece en ambos cadastros (nacional y extranjera), se
  prefiere la activa.

- **`atraso_fora_de_faixa` como columna de auditoría**: en lugar de simplemente
  anular la métrica, se registra el motivo en una columna booleana. Quien
  analiza puede contar cuántas filas tienen métrica anulada y por qué.

- **`pais_aeroporto` por prefijo ICAO**: `RLIKE '^S[BDIJNSW]'` clasifica Brasil
  vs Exterior. Esto es interpretación (regla de negocio), no aritmética — por
  eso está en la Gold, no en la Silver.

### 4. Por qué esta estrategia es adecuada en Medallion

La Gold complementa la Silver con precisión arquitectónica:

- **Silver preserva, Gold decide.** La Silver no tenía `partida_pontual` (umbral
  de 15 minutos) ni `escopo_voo` (clasificación doméstico/internacional). En la
  Gold, ambas son una línea de SQL. Cambiar el umbral de 15 a 30 minutos no
  requiere reprocesar la Silver — solo la Gold.

- **Trazabilidad completa con lineage automático.** Sin escribir código de
  trazabilidad, el Unity Catalog registró la cadena: `archivo CSV → bronze.vra
  → silver.vra → gold.fato_voos → gold.obt_voos`. Esto responde "si muevo aquí,
  ¿qué se rompe allá?" y "¿de dónde viene este número?".

- **Separación fato/dimensión/OBT.** El fato mantiene el modelo estrella (para
  consumo BI tradicional), mientras la OBT desnormaliza todo para consumo por
  IA. La tag `consumo` distingue `bi` de `genie`, señalando el consumidor de
  cada tabla.

- **Cuarentena documentada, no oculta.** Las 5 decisiones de cuarentena están
  comentadas en el SQL del fato_voos, con conteos exactos y justificación de
  negocio. Quien herede el pipeline sabe qué se eliminó, qué se mantuvo y por
  qué.

- **Validación de metadatos como activo.** Cuando el consumidor es un LLM, el
  `COMMENT` es requisito funcional, no decoración. La revisión manual de
  descripciones generadas por IA — validando contra el dato, no contra la
  frase — es un paso de calidad que la mayoría de los pipelines omite.

### 5. Posibles mejoras

1. **Aserción automática del grano.** La diferencia esperada de 41 filas entre
   Silver y Gold debería validarse con `assert` programático, no inspección
   visual. Si la ANAC cambia el patrón de duplicación, el pipeline debe fallar.

2. **Métrica de cobertura del join.** `dim_aeroporto` cubre 100% del fato, pero
   no hay validación automática de esto. Un `SELECT COUNT(*) FROM fato_voos f
   LEFT JOIN dim_aeroporto d ON ... WHERE d.icao_aeroporto IS NULL` con resultado
   esperado = 0 aseguraría la cobertura.

3. **Cuantificación del `atraso_fora_de_faixa`.** La columna existe pero no hay
   un widget o métrica de monitoreo. Un conteo mensual de filas con métrica
   anulada detectaría si la calidad de la fuente empeora con el tiempo.

4. **Versionamento de la regla de pontualidad.** El umbral de 15 minutos está
   harcoded en el SQL. Parametrizarlo (even as a SQL variable) facilitaría
   atender clientes con criterios distintos sin editar el SQL.

5. **Detección de duplicación incremental.** La deduplication actual es
   `CREATE OR REPLACE` — reconstruye todo. En un escenario incremental, un hash
   determinístico por fila permitiría detectar duplicados nuevos sin reescanear
   la tabla completa.

6. **Validación de integridad referencial.** El `LEFT JOIN` en la OBT no
   garantiza que cada `icao_origem` del fato exista en `dim_aeroporto`. Aunque
   hoy la dimensión se construye a partir del fato, si se reconstruye la
   dimensión en un momento distinto del fato, puede haber desincronización.

In [0]:
%sql
SELECT
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END AS categoria,
  COUNT(*) AS quantidade
FROM voebem.gold.obt_voos
GROUP BY
  CASE
    WHEN minutos_recuperados > 0
         AND atraso_chegada_min > 15
      THEN 'Recuperou tempo, mas chegou >15 min atrasado'

    WHEN minutos_recuperados > 0
         AND atraso_chegada_min <= 15
      THEN 'Recuperou tempo e chegou <=15 min atrasado'

    WHEN minutos_recuperados <= 0
      THEN 'Não recuperou tempo'

    ELSE 'Sem informação'
  END
ORDER BY quantidade DESC;

Databricks visualization. Run in Databricks to view.